## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [8]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [9]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [24]:
retriever.invoke("Who won the IIOTY award?")

[Document(id='43ca6b66-0a10-4808-951c-fe89431c39b3', metadata={'source': 'knowledge-base\\employees\\Alex Chen.md', 'doc_type': 'employees'}, page_content='## Annual Performance History\n- **2020:**  \n  - Completed onboarding successfully.  \n  - Met expectations in delivering project milestones.  \n  - Received positive feedback from the team leads.\n\n- **2021:**  \n  - Achieved a 95% success rate in project delivery timelines.  \n  - Awarded "Rising Star" at the annual company gala for outstanding contributions.  \n\n- **2022:**  \n  - Exceeded goals by optimizing existing backend code, improving system performance by 25%.  \n  - Conducted training sessions for junior developers, fostering knowledge sharing.  \n\n- **2023:**  \n  - Led a major overhaul of the API internal architecture, enhancing security protocols.  \n  - Contributed to the company’s transition to a cloud-based infrastructure.  \n  - Received an overall performance rating of 4.8/5.'),
 Document(id='ba5d90aa-b0c4-41

In [11]:
llm.invoke("Who is Avery?")

AIMessage(content='Could you please provide more context or specify which Avery you are referring to? There are many individuals and characters named Avery.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_7f8eb7d1f9', 'id': 'chatcmpl-CvNAoh7sPxxZpPKqnftkLhS1uy8Ek', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--1d3e10de-0ff0-4e4e-8d76-1a90644b7d09-0', usage_metadata={'input_tokens': 11, 'output_tokens': 24, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Time to put this together!

In [14]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [15]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [27]:
answer_question("Who won the prestigious IIOTY award?", [])

'Maxine was recognized as the Insurellm Innovator of the Year (IIOTY) in 2023.'

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!